In [11]:
import requests
import base64
import pandas as pd
import certifi
import urllib3
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from scipy.optimize import linear_sum_assignment
import os
from dotenv import load_dotenv
import certifi
import urllib3

load_dotenv()

http = urllib3.PoolManager(
    cert_reqs="CERT_REQUIRED",
    ca_certs=certifi.where()
)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [12]:
version = "16.4.1" 
url = f"https://ddragon.leagueoflegends.com/cdn/{version}/data/en_US/champion.json"
name_data = requests.get(url).json()["data"]

name_table = {}
for champ in name_data:
    name_table[champ] = name_data[champ]["name"]

num_to_lane = {
    0:"top",
    1:"jng",
    2:"mid",
    3:"bot",
    4:"sup",
    5:"top",
    6:"jng",
    7:"mid",
    8:"bot",
    9:"sup"
}

side_map = {100:"Blue", 200:"Red"}

name_table["FiddleSticks"] = "Fiddlesticks"

name_data_keyed = {v['name']: v for v in name_data.values()}
name_to_id = { champ["name"]: int(champ["key"]) for champ in name_data_keyed.values() }
id_to_name = { int(champ["key"]): champ["name"] for champ in name_data_keyed.values() }

import time

days = 14
cutoff = int((time.time() - days * 24 * 60 * 60) * 1000)

api_key = os.getenv("RIOT_API_KEY")

In [13]:
games = []
 
for c in range(1, 2):
    try:

        queue = "RANKED_SOLO_5x5"
        tier = "CHALLENGER"
        division = "I"

        url = f"https://na1.api.riotgames.com/lol/league-exp/v4/entries/{queue}/{tier}/{division}?page={c}"

        headers = {
            "X-Riot-Token":api_key
        }

        response = http.request("GET", url, headers=headers)

        rank_data = response.json()

        rank_data = rank_data[0: 4]
        matches = []
        for player in rank_data:
            
            puuid = player["puuid"]

            url = f"https://americas.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?start=0&count=20"

            headers = {
                "X-Riot-Token":api_key
            }

            response = http.request("GET", url, headers=headers)

            player_data = response.json()
            matches = matches + player_data
            matches = list(set(matches))

        for matchId in matches:
            url = f"https://americas.api.riotgames.com/lol/match/v5/matches/{matchId}"
            headers = {
                "X-Riot-Token":api_key
            }

            response = http.request("GET", url, headers=headers)
            match_data = response.json()

            # if "info" not in match_data:
            #     print(match_data)
            #     continue

            # if match_data["info"]["queueId"] != 420:
            #     continue

            if match_data["info"]["gameDuration"] < 300 or match_data["info"]["gameCreation"] < cutoff or match_data["info"]["queueId"] != 420 or "info" not in match_data:
                continue
            
            # if match_data["info"]["gameCreation"] < cutoff:
            #     continue

            participants = match_data["info"]["participants"]
            gameid = match_data["metadata"]["matchId"]
            patch = float(match_data["info"]["gameVersion"].split(".")[0] + "." + match_data["info"]["gameVersion"].split(".")[1])

            blue_team = next(t for t in match_data["info"]["teams"] if t["teamId"] == 100)
            red_team  = next(t for t in match_data["info"]["teams"] if t["teamId"] == 200)


            side1 = {"patch":patch, "gameid":match_data["metadata"]["matchId"], "result":int(match_data["info"]["teams"][0]["win"]), "side":side_map[participants[0]["teamId"]], "firstPick":1 if participants[0]["teamId"] == 100 else 0}
            side2 = {"patch":patch, "gameid":match_data["metadata"]["matchId"], "result":int(match_data["info"]["teams"][1]["win"]), "side":side_map[participants[5]["teamId"]], "firstPick":1 if participants[5]["teamId"] == 100 else 0}

            i = 0
            for ban in match_data["info"]["teams"][0]["bans"]:
                i += 1
                if ban["championId"] != -1:
                    side1[f"ban{i}"] = (id_to_name[ban["championId"]])
            i = 0
            for ban in match_data["info"]["teams"][1]["bans"]:
                i += 1
                if ban["championId"] != -1:
                    side2[f"ban{i}"] = (id_to_name[ban["championId"]])

            for i in range(0, 5):
                champ = name_table[participants[i]["championName"]]
                opp_champ = name_table[participants[i+5]["championName"]]
                side1[num_to_lane[i]] = champ
                side1[f"opp_{num_to_lane[i]}"] = opp_champ
            for i in range(5, 10):
                champ = name_table[participants[i]["championName"]]
                opp_champ = name_table[participants[i-5]["championName"]]
                side2[num_to_lane[i]] = champ
                side2[f"opp_{num_to_lane[i]}"] = opp_champ

            games.append(side1)
            games.append(side2)
    except Exception as e:
        print("Error on match:", matchId, e)
        continue

matches_df = pd.DataFrame(games, columns=["patch", 'gameid',"result", 'date','side',"firstPick", 'top','jng','mid','bot','sup', "ban1", "ban2", "ban3", "ban4", "ban5", 'opp_top','opp_jng','opp_mid','opp_bot','opp_sup'])
matches_df.to_csv(r"C:\Users\nikhi\OneDrive\Desktop\ranked_dataset.csv")

In [14]:
matches_df

,patch,gameid,result,date,side,firstPick,top,jng,mid,bot,...,ban1,ban2,ban3,ban4,ban5,opp_top,opp_jng,opp_mid,opp_bot,opp_sup
0,16.4,NA1_5503521179,1,NaN,Blue,1,Jayce,Brand,Riven,Jhin,...,Azir,Lulu,Shaco,Kassadin,Zac,Vayne,Sylas,Zed,Kai'Sa,Neeko
1,16.4,NA1_5503521179,0,NaN,Red,0,Vayne,Sylas,Zed,Kai'Sa,...,Hwei,Bel'Veth,Pyke,Xin Zhao,Nautilus,Jayce,Brand,Riven,Jhin,Vel'Koz
2,16.3,NA1_5495474007,1,NaN,Blue,1,Jax,Hecarim,Kayle,Smolder,...,Poppy,Jayce,Taliyah,Rumble,Orianna,Aurora,Ambessa,Sylas,Senna,Rakan
3,16.3,NA1_5495474007,0,NaN,Red,0,Aurora,Ambessa,Sylas,Senna,...,Fiora,Qiyana,Nunu & Willump,Riven,Rengar,Jax,Hecarim,Kayle,Smolder,Alistar
4,16.4,NA1_5495825836,0,NaN,Blue,1,Ambessa,Vi,Katarina,Vladimir,...,Riven,Zed,Fiddlesticks,Sona,Rengar,Jax,Kindred,Jayce,Ezreal,Bard
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,16.4,NA1_5503536978,0,NaN,Red,0,Mordekaiser,Ivern,Zeri,Aphelios,...,Azir,Fiora,Rengar,Poppy,Blitzcrank,Aatrox,Sylas,Riven,Corki,Morgana
80,16.3,NA1_5494524249,0,NaN,Blue,1,Irelia,Kha'Zix,Galio,Kai'Sa,...,Fiddlesticks,Mel,Caitlyn,Akshan,Gwen,Corki,Rengar,Syndra,Yunara,Thresh
81,16.3,NA1_5494524249,1,NaN,Red,0,Corki,Rengar,Syndra,Yunara,...,Yasuo,Jayce,Ambessa,Riven,Caitlyn,Irelia,Kha'Zix,Galio,Kai'Sa,Pyke
82,16.3,NA1_5492818186,1,NaN,Blue,1,Riven,Jayce,Swain,Vel'Koz,...,Ambessa,Sylas,Fiddlesticks,Nunu & Willump,Viktor,Kled,Shaco,Xerath,Yunara,Lulu


In [2]:
prob_df = matches_df[['top','jng','mid','bot','sup']].melt(var_name='position', value_name="champion")

NameError: name 'matches_df' is not defined

In [7]:
matches_df = pd.read_csv("ranked_dataset.csv").drop(columns=["Unnamed: 0"])

In [8]:
url = f"https://americas.api.riotgames.com/lol/match/v5/matches/{"NA1_5498225932"}"
headers = {
       "X-Riot-Token":api_key
}

response = http.request("GET", url, headers=headers)
match_data = response.json()

NameError: name 'api_key' is not defined

In [9]:
all_champs = pd.concat([matches_df[["top", "jng", "mid", "bot", "sup"]], matches_df[["ban1", "ban2", "ban3", "ban4", "ban5"]]], axis=0).values.ravel()
champion_list = (pd.Series(all_champs).dropna().unique().tolist())
roles = ['top', "jng", "mid", "bot", "sup"]
ban_cols = [f'ban{i}' for i in range(1, 6)]

matches_df = matches_df[matches_df["patch"].astype(str).str.startswith("16")]
# matches_df["result"] = np.random.randint(0, 2, size=len(matches_df))
matches_df["weight"] = (matches_df["patch"].astype(str).str.split(".").str[1].astype(int) * 10)

matches_df[[f'opp_{c}' for c in ban_cols]] = (
    matches_df.groupby('gameid')[ban_cols].transform(lambda x: x.iloc[::-1].values)
)

In [10]:
new_cols = {}

for role in roles:
    for champ in champion_list:
        new_cols[f'pick_{champ}_{role}'] = (
            matches_df[roles].eq(champ).any(axis=1).astype(int)
            - matches_df[[f'opp_{r}' for r in roles]].eq(champ).any(axis=1).astype(int)
        )

for champ in champion_list:
    new_cols[f'ban_{champ}'] = (
        matches_df[ban_cols].eq(champ).any(axis=1)
        | matches_df[[f'opp_{c}' for c in ban_cols]].eq(champ).any(axis=1)
    ).astype(int)

new_features = pd.DataFrame(new_cols, index=matches_df.index)


matches_df = pd.concat([matches_df, new_features], axis=1)
matches_df['side'] = matches_df['side'].map({'Red':0, 'Blue':1})
df_model_lanes = matches_df.drop(columns=roles + ban_cols + [f'opp_{c}' for c in roles] + [f'opp_{c}' for c in ban_cols])

In [ ]:
# for role in roles:
#     for champ in champion_list:
#         matches_df[f'pick_{champ}_{role}'] = (
#             matches_df[roles].eq(champ).any(axis=1).astype(int) - matches_df[[f'opp_{r}' for r in roles]].eq(champ).any(axis=1).astype(int)
#         )

# for champ in champion_list:
#     matches_df[f'ban_{champ}'] = (
#         matches_df[ban_cols].eq(champ).any(axis=1) | matches_df[[f'opp_{c}' for c in ban_cols]].eq(champ).any(axis=1)
#     ).astype(int)

# matches_df['side'] = matches_df['side'].map({'Red':0, 'Blue':1})

# df_model_lanes = matches_df.drop(columns=roles + ban_cols + [f'opp_{c}' for c in roles] + [f'opp_{c}' for c in ban_cols])

In [11]:
matches_df[(matches_df["top"] == "Yorick") & (matches_df["opp_top"] == "Kennen")]

,patch,gameid,result,date,side,firstPick,top,jng,mid,bot,...,ban_K'Sante,ban_Soraka,ban_Trundle,ban_Lillia,ban_Evelynn,ban_Amumu,ban_Tahm Kench,ban_Shyvana,ban_Quinn,ban_Warwick


In [17]:
x = df_model_lanes.drop(columns=['result', 'gameid', "date"])
y = df_model_lanes['result']
groups = df_model_lanes['gameid']

gss = GroupShuffleSplit(test_size = 0.05, n_splits=4)
train_idx, test_idx = next(gss.split(x, y, groups))

x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
weights = x_train['weight']
x_train = x_train.drop(columns='weight')
y_train = y_train.drop(columns='weight')
x_test = x_test.drop(columns='weight')
y_test = y_test.drop(columns='weight')

In [18]:
forest_mod = RandomForestClassifier(
    n_estimators=1200, 
    criterion='gini',
    max_depth=3, 
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    n_jobs=-1)

forest_mod.fit(x_train, y_train, sample_weight=weights)

preds = forest_mod.predict_proba(x_test)[:, 1]
preds_df = df_model_lanes.iloc[x_test.index][['gameid', 'result']].copy()
preds_df['p_win'] = preds

match_acc = (
    preds_df.groupby('gameid').apply(lambda g: g.loc[g['p_win'].idxmax(), 'result'], include_groups = False).mean()
)

match_acc

np.float64(0.5757575757575758)

In [ ]:
with open(r"C:\Riot Games\League of Legends\lockfile", "r") as f:
    parts = f.read().split(":")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Riot Games\\League of Legends\\lockfile'

In [ ]:
with open(r"C:\Riot Games\League of Legends\lockfile", "r") as f:
    parts = f.read().split(":")

port = parts[2]
password = parts[3]

auth = base64.b64encode(f"riot:{password}".encode()).decode()

url = f"https://127.0.0.1:{port}/lol-champ-select/v1/session"

headers = {
    "Authorization": f"Basic {auth}"
}

http = urllib3.PoolManager(cert_reqs='CERT_NONE')
response = http.request("GET", url, headers=headers)

data = response.json()

In [ ]:
response = http.request("GET", url, headers=headers)
match_data = response.json()

In [ ]:
data = {'actions': [[{'actorCellId': 0,
    'championId': 360,
    'completed': True,
    'duration': 0,
    'id': 0,
    'isAllyAction': True,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ban'},
   {'actorCellId': 1,
    'championId': 117,
    'completed': True,
    'duration': 0,
    'id': 1,
    'isAllyAction': True,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ban'},
   {'actorCellId': 2,
    'championId': 121,
    'completed': True,
    'duration': 0,
    'id': 2,
    'isAllyAction': True,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ban'},
   {'actorCellId': 3,
    'championId': -1,
    'completed': True,
    'duration': 0,
    'id': 3,
    'isAllyAction': True,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ban'},
   {'actorCellId': 4,
    'championId': 90,
    'completed': True,
    'duration': 0,
    'id': 4,
    'isAllyAction': True,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ban'},
   {'actorCellId': 5,
    'championId': 50,
    'completed': True,
    'duration': 0,
    'id': 5,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ban'},
   {'actorCellId': 6,
    'championId': 17,
    'completed': True,
    'duration': 0,
    'id': 6,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ban'},
   {'actorCellId': 7,
    'championId': 35,
    'completed': True,
    'duration': 0,
    'id': 7,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ban'},
   {'actorCellId': 8,
    'championId': 3,
    'completed': True,
    'duration': 0,
    'id': 8,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ban'},
   {'actorCellId': 9,
    'championId': 29,
    'completed': True,
    'duration': 0,
    'id': 9,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ban'}],
  [{'actorCellId': -1,
    'championId': 0,
    'completed': True,
    'duration': 0,
    'id': 100,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'ten_bans_reveal'}],
  [{'actorCellId': 0,
    'championId': 51,
    'completed': True,
    'duration': 0,
    'id': 10,
    'isAllyAction': True,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'pick'}],
  [{'actorCellId': 5,
    'championId': 267,
    'completed': True,
    'duration': 0,
    'id': 11,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'pick'},
   {'actorCellId': 6,
    'championId': 21,
    'completed': True,
    'duration': 0,
    'id': 12,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'pick'}],
  [{'actorCellId': 1,
    'championId': 99,
    'completed': True,
    'duration': 0,
    'id': 13,
    'isAllyAction': True,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'pick'},
   {'actorCellId': 2,
    'championId': 19,
    'completed': True,
    'duration': 0,
    'id': 14,
    'isAllyAction': True,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'pick'}],
  [{'actorCellId': 7,
    'championId': 24,
    'completed': True,
    'duration': 0,
    'id': 15,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'pick'},
   {'actorCellId': 8,
    'championId': 84,
    'completed': True,
    'duration': 0,
    'id': 16,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'pick'}],
  [{'actorCellId': 3,
    'championId': 266,
    'completed': False,
    'duration': 0,
    'id': 17,
    'isAllyAction': True,
    'isInProgress': True,
    'pickTurn': 0,
    'type': 'pick'},
   {'actorCellId': 4,
    'championId': 245,
    'completed': True,
    'duration': 0,
    'id': 18,
    'isAllyAction': True,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'pick'}],
  [{'actorCellId': 9,
    'championId': 0,
    'completed': False,
    'duration': 0,
    'id': 19,
    'isAllyAction': False,
    'isInProgress': False,
    'pickTurn': 0,
    'type': 'pick'}]],
 'allowBattleBoost': False,
 'allowDuplicatePicks': False,
 'allowLockedEvents': False,
 'allowPlayerPickSameChampion': False,
 'allowRerolling': False,
 'allowSkinSelection': True,
 'allowSubsetChampionPicks': False,
 'bans': {'myTeamBans': [], 'numBans': 0, 'theirTeamBans': []},
 'benchChampions': [],
 'benchEnabled': False,
 'boostableSkinCount': 0,
 'chatDetails': {'mucJwtDto': {'channelClaim': '1002bca9-67da-4539-b998-55e25800fcdc',
   'domain': 'lol-champ-select',
   'jwt': 'eyJraWQiOiIxIiwiYWxnIjoiUlMyNTYifQ.eyJ0Z3QiOiJuYTEiLCJzdWIiOiJlZWVlYzYzMy0zOTNmLTVhZDAtYjlmZC1iNmYxNzMwN2MzNGYiLCJtZXRhZGF0YSI6eyJnYW1lSWQiOiI1NDk3ODYzNjc5IiwicmVnaW9uIjoiTkExIiwibG9iYnlUeXBlIjoicHJlLWdhbWUiLCJwcm9kdWN0IjoibG9sIn0sImlzcyI6ImxvbC10ZWFtYnVpbGRlciIsImNobiI6IjEwMDJiY2E5LTY3ZGEtNDUzOS1iOTk4LTU1ZTI1ODAwZmNkYyIsInR5cCI6ImxvbC1jaGFtcC1zZWxlY3QiLCJleHAiOjE3NzE3MzQ2MjAsImlhdCI6MTc3MTczNDAyMCwianRpIjoiZTRkYzgwNjItZjY5ZC00N2Q1LTkzNDItYmQ4NDIzODA4NjM2IiwiY3JtIjoiMTAwMmJjYTktNjdkYS00NTM5LWI5OTgtNTVlMjU4MDBmY2RjQGxvbC1jaGFtcC1zZWxlY3QucHZwLm5ldCJ9.mXgINBdhg7tueS0GHG0T1AJB_HVYgrL9D_TFsH7oU-jYdDpWx5USlS2-99mOFu8H9QyrYkaM2ge6WReSOTsKC2JqUID2kSXAVvkP7B4czkxCgszw7JIl6Yde0b4tNGUZRlfe5WyNS7rWH2I4MdKrl1F6Gf_MRH7Zn7stkpsMKHqSkh5XIMQyGwv0RdvWZsN9DjVpd6tfVpB85wIZgXIPJPQt80uvV12-LTNZmWbD2EZCWLTK9NvAl9omf9HBYEoH-LEKgTH0cKreEnQ1pdpHNNz1b7yeUwZ2-_MUJzKWs5lhW3JOc_Fl7MHpKCrDB5Ef2ZRXuAwNq0q9Xujl4f8qZQ',
   'targetRegion': 'na1'},
  'multiUserChatId': '1002bca9-67da-4539-b998-55e25800fcdc',
  'multiUserChatPassword': 'eyJraWQiOiIxIiwiYWxnIjoiUlMyNTYifQ.eyJ0Z3QiOiJuYTEiLCJzdWIiOiJlZWVlYzYzMy0zOTNmLTVhZDAtYjlmZC1iNmYxNzMwN2MzNGYiLCJtZXRhZGF0YSI6eyJnYW1lSWQiOiI1NDk3ODYzNjc5IiwicmVnaW9uIjoiTkExIiwibG9iYnlUeXBlIjoicHJlLWdhbWUiLCJwcm9kdWN0IjoibG9sIn0sImlzcyI6ImxvbC10ZWFtYnVpbGRlciIsImNobiI6IjEwMDJiY2E5LTY3ZGEtNDUzOS1iOTk4LTU1ZTI1ODAwZmNkYyIsInR5cCI6ImxvbC1jaGFtcC1zZWxlY3QiLCJleHAiOjE3NzE3MzQ2MjAsImlhdCI6MTc3MTczNDAyMCwianRpIjoiZTRkYzgwNjItZjY5ZC00N2Q1LTkzNDItYmQ4NDIzODA4NjM2IiwiY3JtIjoiMTAwMmJjYTktNjdkYS00NTM5LWI5OTgtNTVlMjU4MDBmY2RjQGxvbC1jaGFtcC1zZWxlY3QucHZwLm5ldCJ9.mXgINBdhg7tueS0GHG0T1AJB_HVYgrL9D_TFsH7oU-jYdDpWx5USlS2-99mOFu8H9QyrYkaM2ge6WReSOTsKC2JqUID2kSXAVvkP7B4czkxCgszw7JIl6Yde0b4tNGUZRlfe5WyNS7rWH2I4MdKrl1F6Gf_MRH7Zn7stkpsMKHqSkh5XIMQyGwv0RdvWZsN9DjVpd6tfVpB85wIZgXIPJPQt80uvV12-LTNZmWbD2EZCWLTK9NvAl9omf9HBYEoH-LEKgTH0cKreEnQ1pdpHNNz1b7yeUwZ2-_MUJzKWs5lhW3JOc_Fl7MHpKCrDB5Ef2ZRXuAwNq0q9Xujl4f8qZQ'},
 'counter': 54,
 'disallowBanningTeammateHoveredChampions': True,
 'gameId': 5497863679,
 'hasSimultaneousBans': True,
 'hasSimultaneousPicks': False,
 'id': '1002bca9-67da-4539-b998-55e25800fcdc',
 'isCustomGame': False,
 'isLegacyChampSelect': False,
 'isSpectating': False,
 'localPlayerCellId': 3,
 'lockedEventIndex': -1,
 'myTeam': [{'assignedPosition': 'bottom',
   'cellId': 0,
   'championId': 51,
   'championPickIntent': 0,
   'gameName': '',
   'internalName': '',
   'isAutofilled': False,
   'isHumanoid': False,
   'nameVisibilityType': 'HIDDEN',
   'obfuscatedPuuid': '9e4f34ce-0409-0b15-3271-8ca6df4ef52c',
   'obfuscatedSummonerId': 6756267746580,
   'pickMode': 0,
   'pickTurn': 0,
   'playerAlias': '',
   'playerType': '',
   'puuid': '',
   'selectedSkinId': 51019,
   'spell1Id': 4,
   'spell2Id': 21,
   'summonerId': 0,
   'tagLine': '',
   'team': 1,
   'wardSkinId': -1},
  {'assignedPosition': 'utility',
   'cellId': 1,
   'championId': 99,
   'championPickIntent': 0,
   'gameName': '',
   'internalName': '',
   'isAutofilled': False,
   'isHumanoid': False,
   'nameVisibilityType': 'HIDDEN',
   'obfuscatedPuuid': 'fc3ef6d5-6b77-0ea9-2aea-bac02a076e28',
   'obfuscatedSummonerId': 6756279535155,
   'pickMode': 0,
   'pickTurn': 0,
   'playerAlias': '',
   'playerType': '',
   'puuid': '',
   'selectedSkinId': 99018,
   'spell1Id': 4,
   'spell2Id': 14,
   'summonerId': 0,
   'tagLine': '',
   'team': 1,
   'wardSkinId': -1},
  {'assignedPosition': 'jungle',
   'cellId': 2,
   'championId': 19,
   'championPickIntent': 0,
   'gameName': '',
   'internalName': '',
   'isAutofilled': False,
   'isHumanoid': False,
   'nameVisibilityType': 'HIDDEN',
   'obfuscatedPuuid': 'b8279795-ec79-0cc9-19c7-fa7449f7a32d',
   'obfuscatedSummonerId': 2851901679541251,
   'pickMode': 0,
   'pickTurn': 0,
   'playerAlias': '',
   'playerType': '',
   'puuid': '',
   'selectedSkinId': 19000,
   'spell1Id': 4,
   'spell2Id': 11,
   'summonerId': 0,
   'tagLine': '',
   'team': 1,
   'wardSkinId': -1},
  {'assignedPosition': 'top',
   'cellId': 3,
   'championId': 0,
   'championPickIntent': 266,
   'gameName': 'n1x',
   'internalName': '',
   'isAutofilled': False,
   'isHumanoid': False,
   'nameVisibilityType': 'UNHIDDEN',
   'obfuscatedPuuid': '',
   'obfuscatedSummonerId': 0,
   'pickMode': 0,
   'pickTurn': 0,
   'playerAlias': '',
   'playerType': '',
   'puuid': 'eeeec633-393f-5ad0-b9fd-b6f17307c34f',
   'selectedSkinId': 0,
   'spell1Id': 21,
   'spell2Id': 4,
   'summonerId': 122431671,
   'tagLine': 'evil',
   'team': 1,
   'wardSkinId': -1},
  {'assignedPosition': 'middle',
   'cellId': 4,
   'championId': 245,
   'championPickIntent': 245,
   'gameName': '',
   'internalName': '',
   'isAutofilled': False,
   'isHumanoid': False,
   'nameVisibilityType': 'HIDDEN',
   'obfuscatedPuuid': 'c9bca54e-a7b1-055c-1cd1-cf6a14d18605',
   'obfuscatedSummonerId': 6756268482786,
   'pickMode': 0,
   'pickTurn': 0,
   'playerAlias': '',
   'playerType': '',
   'puuid': '',
   'selectedSkinId': 245001,
   'spell1Id': 4,
   'spell2Id': 14,
   'summonerId': 0,
   'tagLine': '',
   'team': 1,
   'wardSkinId': -1}],
 'pickOrderSwaps': [{'cellId': 2, 'id': 24, 'state': 'AVAILABLE'},
  {'cellId': 1, 'id': 25, 'state': 'AVAILABLE'},
  {'cellId': 0, 'id': 26, 'state': 'AVAILABLE'},
  {'cellId': 4, 'id': 27, 'state': 'AVAILABLE'}],
 'positionSwaps': [{'cellId': 2, 'id': 8, 'state': 'AVAILABLE'},
  {'cellId': 1, 'id': 9, 'state': 'AVAILABLE'},
  {'cellId': 0, 'id': 10, 'state': 'AVAILABLE'},
  {'cellId': 4, 'id': 11, 'state': 'AVAILABLE'}],
 'queueId': 420,
 'rerollsRemaining': 0,
 'showQuitButton': False,
 'skipChampionSelect': False,
 'theirTeam': [{'assignedPosition': '',
   'cellId': 5,
   'championId': 267,
   'championPickIntent': 0,
   'gameName': '',
   'internalName': '',
   'isAutofilled': False,
   'isHumanoid': False,
   'nameVisibilityType': 'HIDDEN',
   'obfuscatedPuuid': '',
   'obfuscatedSummonerId': 0,
   'pickMode': 0,
   'pickTurn': 0,
   'playerAlias': '',
   'playerType': '',
   'puuid': '',
   'selectedSkinId': 267000,
   'spell1Id': 0,
   'spell2Id': 0,
   'summonerId': 0,
   'tagLine': '',
   'team': 2,
   'wardSkinId': -1},
  {'assignedPosition': '',
   'cellId': 6,
   'championId': 21,
   'championPickIntent': 0,
   'gameName': '',
   'internalName': '',
   'isAutofilled': False,
   'isHumanoid': False,
   'nameVisibilityType': 'HIDDEN',
   'obfuscatedPuuid': '',
   'obfuscatedSummonerId': 0,
   'pickMode': 0,
   'pickTurn': 0,
   'playerAlias': '',
   'playerType': '',
   'puuid': '',
   'selectedSkinId': 21000,
   'spell1Id': 0,
   'spell2Id': 0,
   'summonerId': 0,
   'tagLine': '',
   'team': 2,
   'wardSkinId': -1},
  {'assignedPosition': '',
   'cellId': 7,
   'championId': 24,
   'championPickIntent': 0,
   'gameName': '',
   'internalName': '',
   'isAutofilled': False,
   'isHumanoid': False,
   'nameVisibilityType': 'HIDDEN',
   'obfuscatedPuuid': '',
   'obfuscatedSummonerId': 0,
   'pickMode': 0,
   'pickTurn': 0,
   'playerAlias': '',
   'playerType': '',
   'puuid': '',
   'selectedSkinId': 24000,
   'spell1Id': 0,
   'spell2Id': 0,
   'summonerId': 0,
   'tagLine': '',
   'team': 2,
   'wardSkinId': -1},
  {'assignedPosition': '',
   'cellId': 8,
   'championId': 84,
   'championPickIntent': 0,
   'gameName': '',
   'internalName': '',
   'isAutofilled': False,
   'isHumanoid': False,
   'nameVisibilityType': 'HIDDEN',
   'obfuscatedPuuid': '',
   'obfuscatedSummonerId': 0,
   'pickMode': 0,
   'pickTurn': 0,
   'playerAlias': '',
   'playerType': '',
   'puuid': '',
   'selectedSkinId': 84000,
   'spell1Id': 0,
   'spell2Id': 0,
   'summonerId': 0,
   'tagLine': '',
   'team': 2,
   'wardSkinId': -1},
  {'assignedPosition': '',
   'cellId': 9,
   'championId': 0,
   'championPickIntent': 0,
   'gameName': '',
   'internalName': '',
   'isAutofilled': False,
   'isHumanoid': False,
   'nameVisibilityType': 'HIDDEN',
   'obfuscatedPuuid': '',
   'obfuscatedSummonerId': 0,
   'pickMode': 0,
   'pickTurn': 0,
   'playerAlias': '',
   'playerType': '',
   'puuid': '',
   'selectedSkinId': 0,
   'spell1Id': 0,
   'spell2Id': 0,
   'summonerId': 0,
   'tagLine': '',
   'team': 2,
   'wardSkinId': -1}],
 'timer': {'adjustedTimeLeftInPhase': 23323,
  'internalNowInEpochMs': 1771734124411,
  'isInfinite': False,
  'phase': 'BAN_PICK',
  'totalTimeInPhase': 25000},
 'trades': [{'cellId': 2, 'id': 33, 'state': 'INVALID'},
  {'cellId': 1, 'id': 41, 'state': 'INVALID'},
  {'cellId': 4, 'id': 51, 'state': 'INVALID'},
  {'cellId': 0, 'id': 25, 'state': 'INVALID'}]}

In [ ]:
lookup = {}
for team_key in ["myTeam", "theirTeam"]:
    for player in data[team_key]:
        
        if player["championId"] == 0:
            continue

        lookup[player["championId"]] = {
            "lane": player["assignedPosition"],
            "team": player["team"]
        }

draft = {}

if data["actions"][2][0]["championId"] in lookup:
    draft["1_pick"] = {"champ":id_to_name[data["actions"][2][0]["championId"]], "lane":lookup[data["actions"][2][0]["championId"]]["lane"], "team":lookup[data["actions"][2][0]["championId"]]["team"], "tags":name_data_keyed[id_to_name[data["actions"][2][0]["championId"]]]["tags"]}

for i in range(3, 7):
    for j in range(0, 2):
        if data["actions"][i][j]["championId"] in lookup:
            draft[f"{i - 1 + j}_pick"] = {"champ":id_to_name[data["actions"][i][j]["championId"]], "lane":lookup[data["actions"][i][j]["championId"]]["lane"], "team":lookup[data["actions"][i][j]["championId"]]["team"], "tags":name_data_keyed[id_to_name[data["actions"][i][j]["championId"]]]["tags"]}

if data["actions"][7][0]["championId"] in lookup:
    draft["1_pick"] = {"champ":id_to_name[data["actions"][7][0]["championId"]], "lane":lookup[data["actions"][7][0]["championId"]]["lane"], "team":lookup[data["actions"][7][0]["championId"]]["team"], "tags":name_data_keyed[id_to_name[data["actions"][7][0]["championId"]]]["tags"]}

In [ ]:
roles
min_games = 10

role_tags = {
    "Fighter":  {"top": 0.5,  "jng": 0.3,  "mid": 0.1,  "bot": 0.05, "sup": 0.05},
    "Tank":     {"top": 0.4,  "jng": 0.2,  "mid": 0.05, "bot": 0.05, "sup": 0.3},
    "Mage":     {"top": 0.05, "jng": 0.05, "mid": 0.6,  "bot": 0.05, "sup": 0.25},
    "Assassin": {"top": 0.1,  "jng": 0.35, "mid": 0.5,  "bot": 0.03, "sup": 0.02},
    "Marksman": {"top": 0.02, "jng": 0.05, "mid": 0.05, "bot": 0.85, "sup": 0.03},
    "Support":  {"top": 0.02, "jng": 0.03, "mid": 0.05, "bot": 0.05, "sup": 0.85},
}

def build_prob_lookup(df=prob_df):
    role_counts = df.groupby(['champion', 'position']).size().unstack(fill_value=0).reindex(columns=roles, fill_value=0)
    role_probs = role_counts.div(role_counts.sum(axis=1), axis=0)
    return role_counts, role_probs.to_dict(orient='index')

def get_tag_prior(tags):
    matching = [role_tags[tag] for tag in tags if tag in role_tags]

    if not matching:
        return {r: 1/len(roles) for r in roles}

    blended = {
        r: np.mean([m[r] for m in matching])
        for r in roles
    }

    total = sum(blended.values())
    return {r: blended[r] / total for r in roles}

def blend_probs(champ, tags, role_counts, prob_lookup):
    prior = get_tag_prior(tags)

    if champ not in role_counts.index:
        return prior

    n = role_counts.loc[champ].sum()
    weight = min(n / min_games, 1)

    data_prob = prob_lookup.get(champ, {r: 0 for r in roles})

    blended = {
        r: weight * data_prob.get(r, 0) + (1 - weight) * prior[r]
        for r in roles
    }

    total = sum(blended.values())
    return {r: blended[r] / total for r in roles}

def assign_enemy_roles(picks_dict, your_team_id, role_counts, prob_lookup):

    norm = {
        "bottom": "bot",
        "utility": "sup",
        "middle": "mid",
        "jungle": "jng",
    }

    for data in picks_dict.values():
        data["lane"] = norm.get(data["lane"], data["lane"])

    enemy_unknown = [
        (pick_id, data)
        for pick_id, data in picks_dict.items()
        if data["team"] != your_team_id and data["lane"] == ""
    ]

    if not enemy_unknown:
        return picks_dict

    enemy_taken_roles = {
        data["lane"]
        for data in picks_dict.values()
        if data["team"] != your_team_id and data["lane"] != ""
    }

    remaining_roles = [r for r in roles if r not in enemy_taken_roles]

    if len(enemy_unknown) > len(remaining_roles):
        raise ValueError("More unknown champs than available roles.")

    prob_matrix = []

    for _, data in enemy_unknown:
        champ = data["champ"]
        tags = data["tags"]

        probs = blend_probs(champ, tags, role_counts, prob_lookup)
        prob_matrix.append([probs[r] for r in remaining_roles])

    prob_matrix = np.array(prob_matrix)

    row_ind, col_ind = linear_sum_assignment(1 - prob_matrix)

    for r, c in zip(row_ind, col_ind):
        pick_id = enemy_unknown[r][0]
        picks_dict[pick_id]["lane"] = remaining_roles[c]

    return picks_dict

In [ ]:
slice_draft = assign_enemy_roles(draft, data["myTeam"][0]["team"], build_prob_lookup(prob_df)[0], build_prob_lookup(prob_df)[1])

In [ ]:
url = f"https://127.0.0.1:{port}/lol-patch/v1/game-version"

response = requests.get(url, auth=('riot', password), verify=False)
patch = response.json()

float(".".join(patch.split(".")[0:2]))

16.4

In [ ]:
draft = {
"team1" : {
    "patch":float(".".join(patch.split(".")[0:2])), 
    "side":"Blue", 
    "firstPick":1, 
    "Picks":{
        "top": None, 
        "jng": None, 
        "mid": None, 
        "bot": None, 
        "sup": None}, 
    "Bans":[]},
}

In [ ]:
draft["team2"] = {
    "patch":draft["team1"]["patch"], 
    "side":"Red" if draft["team1"]["side"] == "Blue" else "Blue", 
    "firstPick":1 if draft["team1"]["firstPick"] == 0 else 0, 
    "Picks":{
        "top": None, 
        "jng": None, 
        "mid": None, 
        "bot": None, 
        "sup": None}, 
    "Bans":[]}


In [ ]:
for i in range(0, 10):
    if i < 5:
        banned_champ = data["actions"][0][i]["championId"]
        if banned_champ == -1:
            draft["team1"]["Bans"].append(None)
            continue
        draft["team1"]["Bans"].append(id_to_name[banned_champ])
    else:
        banned_champ = data["actions"][0][i]["championId"]
        if banned_champ == -1:
            draft["team2"]["Bans"].append(None)
            continue
        draft["team2"]["Bans"].append(id_to_name[banned_champ])

bans = draft["team1"]['Bans'] + draft["team2"]['Bans']

In [ ]:
def scale(x):
    y = (x - 0.48) / (0.52 - 0.48)
    return y

def convert(team1=draft["team1"], team2=draft["team2"]):

    dfteam1 = pd.DataFrame({'patch':[team1['patch']], 'side':[team1['side']], 'firstPick':[team1['firstPick']]})
    for set in team1['Picks']:
        dfteam1[f'pick_{team1['Picks'][set]}_{set}'] = 1
    for set in team2['Picks']:
        dfteam1[f'pick_{team2['Picks'][set]}_{set}'] = -1
    for champ in team1['Bans'] + team2['Bans']:
        dfteam1[f'ban_{champ}'] = 1

    dfteam2 = pd.DataFrame({'patch':[team2['patch']], 'side':[team2['side']], 'firstPick':[team2['firstPick']]})
    for set in team2['Picks']:
        dfteam2[f'pick_{team2['Picks'][set]}_{set}'] = 1
    for set in team1['Picks']:
        dfteam2[f'pick_{team1['Picks'][set]}_{set}'] = -1
    for champ in team1['Bans'] + team2['Bans']:
        dfteam2[f'ban_{champ}'] = 1

    df_input = pd.concat([dfteam1, dfteam2], ignore_index=True)
    df_input = df_input.reindex(columns=df_model_lanes.columns.drop(["gameid", "result", "weight", "date"]), fill_value=0)

    df_input['side'] = df_input['side'].map({'Red':0, 'Blue':1})

    return df_input

preds = forest_mod.predict_proba(convert(draft["team1"], draft["team2"]))[:, 1]
print(preds)
print(f'Team 1 Draft Strength: {scale(preds[0]) * 100}')
print(f'Team 2 Draft Strength: {scale(preds[1]) * 100}')

[0.49941782 0.4992258 ]
Team 1 Draft Strength: 48.544557251870245
Team 2 Draft Strength: 48.06449249482726


In [ ]:
empty = {'team1':[], 'team2':[]}

for role in draft["team1"]['Picks']:
    if draft["team1"]['Picks'][role] == None:
        empty['team1'].append(role)

for role in draft["team1"]['Picks']:
    if draft["team2"]['Picks'][role] == None:
        empty['team2'].append(role)

df_input = convert(draft["team1"], draft["team2"])

In [ ]:
dfPro26 = pd.read_csv('2026_LoL_esports_match_data_from_OraclesElixir.csv')
dfPro26 = dfPro26[~(dfPro26['league'].isin(['LPL', "DCup"]))].reset_index()
dfPro26 = dfPro26[['patch', 'gameid', 'date', 'side', 'firstPick', 'position', 'champion','pick1', 'pick2', 'pick3', 'pick4', 'pick5', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5', 'result']].copy()

lookup = dfPro26[dfPro26["position"] != "team"].set_index(["gameid", "champion"])["position"]

team_mask = dfPro26["position"] == "team"


lookup[lookup.index.duplicated(keep=False)]
for i in range(1, 6):
    keys = list(zip(dfPro26.loc[team_mask, "gameid"], dfPro26.loc[team_mask, f"pick{i}"]))
    dfPro26.loc[team_mask, f"lane_{i}"] = [lookup.get(k) for k in keys]

players = dfPro26[dfPro26["position"] != "team"].copy()
team_rows = dfPro26[dfPro26["position"] == "team"].copy()

def get_lane_pick_map(row):
    return {row[f"lane_{i}"]: i for i in range(1, 6)}


team_rows["lane_pick_map"] = team_rows.apply(get_lane_pick_map, axis=1)
pick_lookup = team_rows.set_index(["gameid", "side"])["lane_pick_map"]

def get_pick_number(row):
    try:
        lane_map = pick_lookup[(row["gameid"], row["side"])]
        return lane_map.get(row["position"])
    except KeyError:
        return 99
    
def get_enemy_pick_number(row):
    try:
        enemy_side = "Red" if row["side"] == "Blue" else "Blue"
        lane_map = pick_lookup[(row["gameid"], enemy_side)]
        return lane_map.get(row["position"])
    except KeyError:
        return 99
team_rows["lane_pick_map"] = team_rows.apply(get_lane_pick_map, axis=1)
pick_lookup = team_rows.set_index(["gameid", "side"])["lane_pick_map"]

players["pick_number"] = players.apply(get_pick_number, axis=1)
players["enemy_pick_number"] = players.apply(get_enemy_pick_number, axis=1)
players["is_blind"] = players["pick_number"] < players["enemy_pick_number"]

In [ ]:
winrates = {}
for role in empty['team1']:
    winrates[f'{role}_winrates'] = {}
    team_pick = draft["team1"]['Picks'][role]
    opp_pick = draft["team2"]['Picks'][role]
    if opp_pick == None:
        for champ in set(players[(players['position'] == role) & players['is_blind'] == True]['champion'].unique()) - set(bans):
            total = players[(players['position'] == role) & players['is_blind'] == True]['champion'].value_counts()
            winrates[f"{role}_winrates"][champ] = total[champ]

    elif opp_pick != None:
        for champ in set(matches_df[role].unique()) - set(bans):
            if len(matches_df[(matches_df[role] == champ) & (matches_df[f'opp_{role}'] == opp_pick)]) != 0:
                win_num = len(matches_df[(matches_df[role] == champ) & (matches_df[f'opp_{role}'] == opp_pick) & (matches_df['result'] == 1)])
                total_games = len(matches_df[(matches_df[role] == champ) & (matches_df[f'opp_{role}'] == opp_pick)])
                most_games = matches_df[(matches_df[f'opp_{role}'] == opp_pick)][role].value_counts().iloc[0]
                winrate = win_num / total_games
                winrate_adj = 0.5 + ((winrate - 0.5) * (total_games / most_games))
                winrates[f"{role}_winrates"][champ] = winrate_adj
df_winrates = pd.DataFrame(winrates).reset_index().rename(columns={"index": "champ"})
missing_cols = [f"{role}_winrates" for role in empty['team1']]
for col in missing_cols:
    print(df_winrates[['champ', col]].head(5))

      champ  top_winrates
0      Gwen          12.0
1    Gragas           1.0
2      Sion          89.0
3  Volibear           2.0
4  Vladimir           1.0
      champ  jng_winrates
0      Gwen           3.0
1    Gragas           NaN
2      Sion           1.0
3  Volibear           3.0
4  Vladimir           NaN
      champ  mid_winrates
0      Gwen           NaN
1    Gragas           1.0
2      Sion           4.0
3  Volibear           NaN
4  Vladimir           NaN
      champ  bot_winrates
0      Gwen           NaN
1    Gragas           NaN
2      Sion           1.0
3  Volibear           NaN
4  Vladimir           NaN
      champ  sup_winrates
0      Gwen           NaN
1    Gragas           NaN
2      Sion           NaN
3  Volibear           NaN
4  Vladimir           NaN


In [ ]:
results = []
Y = pd.DataFrame(columns=df_model_lanes.columns).drop(columns=["gameid", "result", "weight"]).drop(columns=['date'])
for role in empty['team1']:

    for champ in df_winrates.sort_values(f'{role}_winrates', ascending=False).head(10)['champ']:

        draft["team1"]['Picks'][role] = champ


        X = convert(draft["team1"], draft["team2"])
        X = X.drop(columns=[col for col in X.columns if col.startswith('pick_None')])

        preds = forest_mod.predict_proba(X)[:, 1]
        strength = preds[0] - preds[1]
        d = {"Pick Champ":champ, "Role":role, "Strength":strength}
        results.append(d)

results_strength = pd.DataFrame(results)
results_strength.sort_values('Strength', ascending=False)

,Pick Champ,Role,Strength
15,Jayce,jng,0.017910
2,Sion,top,0.004400
1,Ambessa,top,0.002155
5,Zaahen,top,0.001593
16,Ambessa,jng,0.001510
3,Renekton,top,0.000192
7,Gnar,top,0.000192
0,Rumble,top,0.000192
4,K'Sante,top,0.000192
13,Pantheon,jng,0.000192
